# Boundary kernel testing

The runs use `AdvectionRK4_3D` with no bottom/coast boundary kernel, so a
particle that reaches the seafloor (or a masked coastal cell) stops - and a
few even slip *below* the model bathymetry. The subduction that carries Amazon
water down is real physics we want to keep; only the terminal freeze at the
boundary is the artefact.

This is a small controlled experiment: seed 100 particles **on the shallow
Guiana shelf, near the bottom, where the E-group particles subducted and
stuck**, then compare three kernel stacks over a short run:

1. **baseline** - `AdvectionRK4_3D` only (reproduces the freezing / sub-floor
   penetration);
2. **depth-clamp** - a no-normal-flow bottom: never let a particle go below
   `seafloor - buffer`, so it rides the bottom and keeps being advected
   horizontally;
3. **clamp + lateral unbeach** - additionally, if the horizontal velocity is
   zero (a coastal wall), revert to the last wet position so the particle is
   not pinned against land.

Fields are built exactly as `MERCATOR_parcels_TS.ipynb` builds them; the model
bathymetry (`mbathy`) is added as a `bathy` field the kernels can sample.
`ScipyParticle` is used so the kernels are plain, auditable Python.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import xarray as xr
import math
from operator import attrgetter
from datetime import timedelta
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from parcels import (FieldSet, Field, ParticleSet, ScipyParticle, Variable,
                     AdvectionRK4_3D)
import config as C

# --- TUNE ME -----------------------------------------------------------------
MONTHS    = ["2005-06", "2005-07"]      # fields spanning the run
SEED_BOX  = dict(lon=(-53.0, -49.5), lat=(5.0, 8.0), max_levels=15)  # shelf
N_PART    = 100
RUN_DAYS  = 15
DT_MIN    = 20
SEED_ABOVE_BOTTOM = 3.0                 # release this far above the seafloor (m)
BOTTOM_BUFFER     = 2.0                 # clamp keeps particles this far above it
# -----------------------------------------------------------------------------

UVW  = "/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW"
Hgr  = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
MESHZ = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
print("months:", MONTHS, "| seed box:", SEED_BOX, "| N:", N_PART,
      "| run:", RUN_DAYS, "d")

## Build the FieldSet and attach the bathymetry
Same C-grid recipe as the production run. `bathy` is the seafloor depth
(`gdepw_0[mbathy]`), land set to a huge value so the clamp never fires there;
it is added on the velocity grid and time-extrapolated (static field).

In [ ]:
fn = {v: {"data": [f"{UVW}/{v}_{m}{'fc' if v=='W' else 'c'}.nc" for m in MONTHS],
          "lon": Hgr, "lat": Hgr, "depth": f"{UVW}/W_{MONTHS[0]}fc.nc"}
      for v in ("U", "V", "W")}
fieldset = FieldSet.from_netcdf(
    fn, {"U": "vozocrtx", "V": "vomecrty", "W": "vovecrtz"},
    {k: {"lon": "glamf", "lat": "gphif", "depth": "depthw", "time": "time_counter"}
     for k in ("U", "V", "W")},
    interp_method={k: "cgrid_velocity" for k in ("U", "V", "W")}, mesh="spherical")

Zg = xr.open_dataset(MESHZ)
mbathy = np.asarray(Zg["mbathy"]).squeeze().astype(int)
gdepw  = np.asarray(Zg["gdepw_0"]).squeeze()
bottom2d = np.where(mbathy > 0, gdepw[np.clip(mbathy, 0, 49)], 1e10)

G = fieldset.U.grid
bathy = Field("bathy", bottom2d,
              grid=G.__class__(lon=G.lon, lat=G.lat, depth=np.array([0.]),
                               time=np.array([0.]), mesh="spherical"),
              interp_method="nearest")
bathy.allow_time_extrapolation = True
fieldset.add_field(bathy)
fieldset.add_constant("buf", BOTTOM_BUFFER)

grid_tree = cKDTree(np.c_[np.asarray(G.lon).ravel(), np.asarray(G.lat).ravel()])
def seafloor(lat, lon):
    return bottom2d.ravel()[grid_tree.query(np.c_[np.atleast_1d(lon),
                                                  np.atleast_1d(lat)])[1]]
print("fieldset + bathy ready | bathy at (6,-52) =", float(fieldset.bathy[0, 0, 6., -52.]))

## The kernels

In [ ]:
def KeepAboveBottom(particle, fieldset, time):
    """No-normal-flow bottom: never sink below seafloor - buffer."""
    bath = fieldset.bathy[time, 0., particle.lat, particle.lon]
    if particle.depth + particle_ddepth > bath - fieldset.buf:  # noqa
        particle_ddepth = bath - fieldset.buf - particle.depth  # noqa


def UnBeachLateral(particle, fieldset, time):
    """If horizontal flow is zero (a coastal wall), revert to last wet point."""
    (u, v) = fieldset.UV[time, particle.depth, particle.lat, particle.lon]
    if math.fabs(u) < 1e-12 and math.fabs(v) < 1e-12:
        particle_dlon = particle.lon_prev - particle.lon   # noqa
        particle_dlat = particle.lat_prev - particle.lat   # noqa
    else:
        particle.lon_prev = particle.lon
        particle.lat_prev = particle.lat


class BParticle(ScipyParticle):
    lon_prev = Variable("lon_prev", dtype=np.float32, initial=attrgetter("lon"))
    lat_prev = Variable("lat_prev", dtype=np.float32, initial=attrgetter("lat"))

print("kernels defined")

## Seed 100 particles near the bottom in the sticking zone
Random positions inside the shelf box (only wet columns shallower than
`max_levels`), each released `SEED_ABOVE_BOTTOM` m above its local seafloor.

In [ ]:
Hg = xr.open_dataset(Hgr)
glamt = np.asarray(Hg["glamt"]).squeeze(); gphit = np.asarray(Hg["gphit"]).squeeze()
shelf = ((glamt > SEED_BOX["lon"][0]) & (glamt < SEED_BOX["lon"][1])
         & (gphit > SEED_BOX["lat"][0]) & (gphit < SEED_BOX["lat"][1])
         & (mbathy > 0) & (mbathy < SEED_BOX["max_levels"]))
jj, ii = np.where(shelf)
rng = np.random.default_rng(C.RANDOM_STATE)
sel = rng.choice(len(jj), N_PART, replace=len(jj) < N_PART)
lon0 = glamt[jj[sel], ii[sel]]; lat0 = gphit[jj[sel], ii[sel]]
H0 = seafloor(lat0, lon0)
dep0 = np.clip(H0 - SEED_ABOVE_BOTTOM, 1.0, None)
print(f"seeded {N_PART} particles | seafloor {H0.min():.1f}..{H0.max():.1f} m | "
      f"release depth {dep0.min():.1f}..{dep0.max():.1f} m")

## Run the three configurations
Same seed, same fields; only the kernel stack differs. We record the full paths
to compare motion and bottom penetration.

In [ ]:
t0 = fieldset.U.grid.time[0]
configs = {
    "baseline":        [AdvectionRK4_3D],
    "clamp":           [AdvectionRK4_3D, KeepAboveBottom],
    "clamp+unbeach":   [AdvectionRK4_3D, KeepAboveBottom, UnBeachLateral],
}
results = {}
for tag, kern in configs.items():
    ps = ParticleSet(fieldset, pclass=BParticle, lon=lon0.copy(), lat=lat0.copy(),
                     depth=dep0.copy(), time=[t0] * N_PART)
    out = ps.ParticleFile(name=f"bkt_{tag}.zarr",
                          outputdt=timedelta(hours=12))
    ps.execute(kern, runtime=timedelta(days=RUN_DAYS),
               dt=timedelta(minutes=DT_MIN), output_file=out)
    fla = np.array([p.lat for p in ps]); flo = np.array([p.lon for p in ps])
    fd  = np.array([p.depth for p in ps])
    results[tag] = dict(lat=fla, lon=flo, depth=fd,
                        disp=np.hypot(fla - lat0, (flo - lon0)
                                      * np.cos(np.deg2rad(lat0))) * 111.32,
                        below=fd - seafloor(fla, flo))
    print(f"  {tag} done")

## Compare
`horiz displ` = how far it travelled (frozen -> ~0). `below seafloor` > 0 means
it penetrated the bathymetry (should be impossible; baseline lets it happen).

In [ ]:
rows = []
for tag, r in results.items():
    rows.append(dict(config=tag,
                     median_km=np.median(r["disp"]),
                     frozen_pct=100 * np.mean(r["disp"] < 1.0),
                     pct_below_floor=100 * np.mean(r["below"] > 1.0),
                     max_below_m=r["below"].max()))
summary = pd.DataFrame(rows).set_index("config")
print(summary.round(1).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
col = {"baseline": "#e34948", "clamp": "#2a78d6", "clamp+unbeach": "#1baf7a"}
for tag, r in results.items():
    ax[0].hist(r["disp"], bins=30, histtype="step", lw=1.8, color=col[tag], label=tag)
    ax[1].hist(r["below"], bins=30, histtype="step", lw=1.8, color=col[tag], label=tag)
ax[0].set_xlabel("horizontal displacement (km)"); ax[0].set_ylabel("particles")
ax[0].set_title("motion: baseline piles up at zero", loc="left", fontsize=10)
ax[1].axvline(0, color="k", lw=1)
ax[1].set_xlabel("final depth minus seafloor (m)   >0 = below the floor")
ax[1].set_title("bottom penetration", loc="left", fontsize=10)
for a in ax:
    a.legend(frameon=False, fontsize=8)
    for s in ("top", "right"): a.spines[s].set_visible(False)
    a.grid(color="0.92"); a.set_axisbelow(True)
fig.tight_layout(); plt.show()

## Paths - baseline vs clamp
Member tracks coloured by config. Baseline tracks stall; clamp tracks keep
moving along the shelf.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True, sharey=True)
for ax, tag in zip(axes, configs):
    ds = xr.open_zarr(f"bkt_{tag}.zarr")
    lo = ds.lon.values; la = ds.lat.values
    for k in range(lo.shape[0]):
        ax.plot(lo[k], la[k], "-", lw=.6, color=col[tag], alpha=.5)
    ax.plot(lon0, lat0, "k.", ms=3)
    ax.set_title(f"{tag}  (median {results[tag]['disp'].mean():.0f} km)",
                 fontsize=10, loc="left")
    ax.set_xlabel("lon")
axes[0].set_ylabel("lat")
fig.suptitle("black = release; lines = 15-day paths", x=.01, ha="left")
fig.tight_layout(); plt.show()

## Verdict

- **baseline** should show a spike of frozen particles at 0 km and a fraction
  sitting *below* the seafloor.
- **clamp** removes the sub-floor penetration and lets the stuck ones drift
  along the bottom - the physical no-normal-flow behaviour, while the
  subduction that brought them down is untouched.
- **clamp+unbeach** additionally frees any pinned against the coast; on this
  bottom-seeded set it should behave like clamp.

If the clamp median displacement is well above baseline with zero sub-floor
penetration, this is the kernel to add to the production run. The lateral
unbeach matters mainly for the ~13% of stuck particles that are against a
coastal wall rather than the seafloor.
